Imports

In [5]:
!pip install spacy


[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: pip install --upgrade pip


In [7]:
!pip install --upgrade typing_extensions


[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: pip install --upgrade pip


In [1]:
# Download directly using pip
!pip install https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.8.0/en_core_web_sm-3.8.0-py3-none-any.whl

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 528.8 kB/s eta 0:00:0000:0100:01

[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: pip install --upgrade pip


In [4]:
import sys
!{sys.executable} -m pip install groq neo4j

  Using cached groq-0.37.1-py3-none-any.whl.metadata (16 kB)
  Using cached anyio-4.12.0-py3-none-any.whl.metadata (4.3 kB)
  Using cached distro-1.9.0-py3-none-any.whl.metadata (6.8 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached sniffio-1.3.1-py3-none-any.whl.metadata (3.9 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
Using cached groq-0.37.1-py3-none-any.whl (137 kB)
Using cached anyio-4.12.0-py3-none-any.whl (113 kB)
Using cached distro-1.9.0-py3-none-any.whl (20 kB)
Using cached httpx-0.28.1-py3-none-any.whl (73 kB)
Using cached httpcore-1.0.9-py3-none-any.whl (78 kB)
Using cached sniffio-1.3.1-py3-none-any.whl (10 kB)
Using cached h11-0.16.0-py3-none-any.whl (37 kB)

[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: python3.10 -m pip install --upgrade pip


In [5]:
import sys
!{sys.executable} -m pip install sentence_transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.5/74.5 MB 643.4 kB/s eta 0:00:0000:01:020m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 189.7 kB/s eta 0:00:0000:0200:03
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.1/566.1 kB 878.6 kB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 408.9 kB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 781.3 kB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 740.1 kB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 585.7 kB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 536.2/536.2 kB 739.8 kB/s eta 0:00:00:--:--

[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: python3.10 -m pip install --upgrade pip


In [3]:
import sys
!{sys.executable} -m pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 1.3 MB/s  0:00:02 eta 0:00:010m


In [9]:
import sys
!{sys.executable} -m pip install --upgrade pip

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 3.4 MB/s eta 0:00:00a 0:00:01
  Attempting uninstall: pip
    Found existing installation: pip 25.0.1
    Uninstalling pip-25.0.1:
      Successfully uninstalled pip-25.0.1


In [4]:
import re
from typing import Dict, List, Tuple, Optional, Any
from collections import defaultdict
import spacy
nlp = spacy.load('en_core_web_sm')
from spacy.matcher import PhraseMatcher
from enum import Enum
from groq import Groq
from sentence_transformers import SentenceTransformer
from neo4j import GraphDatabase
import numpy as np
from tqdm import tqdm
import faiss

1. INPUT PREPROCESSING

In [5]:
class Intent(Enum):
    """Hotel travel assistant intent types"""
    HOTEL_SEARCH = "HOTEL_SEARCH"
    HOTEL_RECOMMENDATION = "HOTEL_RECOMMENDATION"
    REVIEW_QUERY = "REVIEW_QUERY"
    HOTEL_COMPARISON = "HOTEL_COMPARISON"
    VISA_INQUIRY = "VISA_INQUIRY"
    LOCATION_BASED_QUERY = "LOCATION_BASED_QUERY"
    TRAVELER_PREFERENCE_QUERY = "TRAVELER_PREFERENCE_QUERY"
    RATING_ANALYSIS = "RATING_ANALYSIS"
    STATISTICAL_QUERY = "STATISTICAL_QUERY"
    DEMOGRAPHIC_RECOMMENDATION = "DEMOGRAPHIC_RECOMMENDATION"
    UNKNOWN = "UNKNOWN"

ENTITY EXTRACTION

In [6]:
class EntityExtractor:
    """
    Extract entities from user queries using hybrid approach:
    - spaCy PhraseMatcher for named entities
    - Regex for structured patterns
    """

    def __init__(self, neo4j_driver=None):
        """
        Initialize entity extractor.

        Args:
            neo4j_driver: Optional Neo4j driver to fetch entities from database
        """
        # Load spaCy model
        try:
            self.nlp = spacy.load("en_core_web_sm")
        except OSError:
            print("Downloading spaCy model...")
            import subprocess
            subprocess.run(["python", "-m", "spacy", "download", "en_core_web_sm"])
            self.nlp = spacy.load("en_core_web_sm")

        # Initialize entity lists
        if neo4j_driver:
            # Fetch from Neo4j (production)
            self.cities = self._fetch_cities_from_neo4j(neo4j_driver)
            self.countries = self._fetch_countries_from_neo4j(neo4j_driver)
            self.hotels = self._fetch_hotels_from_neo4j(neo4j_driver)
        else:
            # Use sample lists (development/testing)
            self.cities = [
              'new york', 'nyc', 'london', 'paris', 'tokyo', 'dubai',
              'singapore', 'sydney', 'rio de janeiro', 'rio', 'berlin',
              'toronto', 'shanghai', 'mexico city', 'mumbai', 'rome',
              'cape town', 'seoul', 'moscow', 'cairo', 'barcelona',
              'bangkok', 'istanbul', 'amsterdam', 'buenos aires', 'lagos',
              'wellington'
            ]
            self.countries = [
              'united states', 'usa', 'us','united kingdom', 'uk', 'france', 'japan',
              'united arab emirates', 'uae', 'singapore', 'australia', 'brazil',
              'germany', 'canada', 'china', 'mexico', 'india', 'italy',
              'south africa', 'south korea', 'russia', 'egypt', 'spain',
              'thailand', 'turkey', 'netherlands', 'argentina', 'nigeria',
              'new zealand'
            ]
            self.hotels = [
              'the azure tower', 'the royal compass', "l'étoile palace",
              'kyo-to grand', 'the golden oasis', 'marina bay zenith',
              'sydney harbour grand', 'copacabana lux', 'berlin mitte elite',
              'the maple grove', 'the bund palace', 'aztec heights',
              'the gateway royale', 'colosseum gardens', 'table mountain view',
              'han river oasis', 'kremlin suites', 'nile grandeur',
              "gaudi's retreat", 'the orchid palace', 'the bosphorus inn',
              'canal house grand', 'tango boutique', 'the savannah house',
              'the kiwi grand'
            ]

        # Traveler types (domain knowledge)
        self.traveler_types = [
            'business traveler', 'business', 'solo traveler', 'solo',
            'family', 'couple'
        ]

        # Setup spaCy matchers
        self._setup_matchers()

    def _fetch_cities_from_neo4j(self, driver):
        """Fetch cities from Neo4j database"""
        query = "MATCH (c:City) RETURN DISTINCT c.name as name"
        with driver.session() as session:
            result = session.run(query)
            return [record["name"].lower() for record in result]

    def _fetch_countries_from_neo4j(self, driver):
        """Fetch countries from Neo4j database"""
        query = "MATCH (c:Country) RETURN DISTINCT c.name as name"
        with driver.session() as session:
            result = session.run(query)
            countries = [record["name"].lower() for record in result]

        # Add common abbreviations
        abbreviations = {
            'usa': 'united states',
            'us': 'united states',
            'uk': 'united kingdom',
            'uae': 'united arab emirates'
        }
        countries.extend(abbreviations.keys())
        return countries

    def _fetch_hotels_from_neo4j(self, driver):
        """Fetch hotels from Neo4j database"""
        query = "MATCH (h:Hotel) RETURN DISTINCT h.name as name"
        with driver.session() as session:
            result = session.run(query)
            return [record["name"].lower() for record in result]

    def _setup_matchers(self):
        """Setup PhraseMatcher for named entities"""
        # Create matchers
        self.city_matcher = PhraseMatcher(self.nlp.vocab, attr="LOWER")
        self.country_matcher = PhraseMatcher(self.nlp.vocab, attr="LOWER")
        self.hotel_matcher = PhraseMatcher(self.nlp.vocab, attr="LOWER")
        self.traveler_matcher = PhraseMatcher(self.nlp.vocab, attr="LOWER")

        # Add patterns (sort by length descending to match longest phrases first)
        city_patterns = [self.nlp.make_doc(city) for city in sorted(self.cities, key=len, reverse=True)]
        country_patterns = [self.nlp.make_doc(country) for country in sorted(self.countries, key=len, reverse=True)]
        hotel_patterns = [self.nlp.make_doc(hotel) for hotel in sorted(self.hotels, key=len, reverse=True)]
        traveler_patterns = [self.nlp.make_doc(t) for t in sorted(self.traveler_types, key=len, reverse=True)]

        self.city_matcher.add("CITY", city_patterns)
        self.country_matcher.add("COUNTRY", country_patterns)
        self.hotel_matcher.add("HOTEL", hotel_patterns)
        self.traveler_matcher.add("TRAVELER_TYPE", traveler_patterns)

    def extract_entities(self, text: str) -> Dict[str, List[str]]:
        """
        Extract all entities from text using hybrid approach.

        Args:
            text: User query text

        Returns:
            Dictionary of entity types and their values
        """
        entities = defaultdict(list)

        # Preprocess text
        text_lower = text.lower().strip()

        # Process with spaCy
        doc = self.nlp(text_lower)

        # === SPACY PHRASEMATCHER: For named entities ===

        # Extract cities
        city_matches = self.city_matcher(doc)
        for match_id, start, end in city_matches:
            entities['city'].append(doc[start:end].text)

        # Extract countries
        country_matches = self.country_matcher(doc)
        for match_id, start, end in country_matches:
            entities['country'].append(doc[start:end].text)

        # Extract hotels
        hotel_matches = self.hotel_matcher(doc)
        for match_id, start, end in hotel_matches:
            entities['hotel'].append(doc[start:end].text)

        # Extract traveler types
        traveler_matches = self.traveler_matcher(doc)
        for match_id, start, end in traveler_matches:
            entities['traveler_type'].append(doc[start:end].text)

        # === REGEX: For structured patterns ===

        # Age patterns
        age_pattern = r'\b(\d+)\s*(?:year|yr)s?\s*old\b'
        age_matches = re.findall(age_pattern, text_lower)
        if age_matches:
            entities['age'].extend(age_matches)

        # Star rating patterns
        star_pattern = r'\b(\d)\s*[-\s]?star\b'
        star_matches = re.findall(star_pattern, text_lower)
        if star_matches:
            entities['star_rating'].extend(star_matches)

        # Gender patterns
        if re.search(r'\b(?:male|man|men|boy)\b', text_lower):
            entities['gender'].append('male')
        if re.search(r'\b(?:female|woman|women|girl|lady)\b', text_lower):
            entities['gender'].append('female')

        # Hotel features (using spaCy lemmatization)
        feature_keywords = {
            'clean': 'cleanliness',
            'cleanliness': 'cleanliness',
            'comfort': 'comfort',
            'comfortable': 'comfort',
            'facility': 'facilities',
            'facilities': 'facilities',
            'amenity': 'facilities',
            'amenities': 'facilities',
            'location': 'location',
            'staff': 'staff',
            'service': 'staff',
            'value': 'value_for_money',
            'price': 'value_for_money',
            'money': 'value_for_money'
        }

        for token in doc:
            if token.lemma_ in feature_keywords:
                entities['feature'].append(feature_keywords[token.lemma_])

        # Remove duplicates while preserving order
        for key in entities:
            seen = set()
            entities[key] = [x for x in entities[key] if not (x in seen or seen.add(x))]

        return dict(entities)

INTENT CLASSIFICATION

LLM Based

In [7]:
client = Groq(api_key="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")


# --- Prompt ---
HOTEL_INTENT_PROMPT = """
You are an intent classifier for a Hotel Travel Assistant.
Your task: read the user query and return ONLY ONE intent label from this list:

HOTEL_SEARCH
HOTEL_RECOMMENDATION
REVIEW_QUERY
HOTEL_COMPARISON
VISA_INQUIRY
LOCATION_BASED_QUERY
TRAVELER_PREFERENCE_QUERY
RATING_ANALYSIS
STATISTICAL_QUERY
DEMOGRAPHIC_RECOMMENDATION

Rules:
- Return ONLY the label.
- No sentences.
- No explanation.
- No JSON.

User Query: "{query}"
"""


# --- Classifier function ---
def classify_hotel_intent(query: str) -> str:
    prompt = HOTEL_INTENT_PROMPT.format(query=query)

    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
        max_tokens=10
    )

    return response.choices[0].message.content.strip()

In [8]:
# --- TESTS ---
print(classify_hotel_intent("Find hotels in Paris"))
print(classify_hotel_intent("Recommend a hotel for a 28-year-old female solo traveler"))
print(classify_hotel_intent("Do I need a visa to travel from Egypt to France?"))
print(classify_hotel_intent("What do people say about the Hilton in Cairo?"))
print(classify_hotel_intent("Compare hotels in Rome vs Florence"))
print(classify_hotel_intent("What's the average rating of 5-star hotels?"))
print(classify_hotel_intent("Show me hotels with good cleanliness ratings in Dubai"))
print(classify_hotel_intent("Which hotels do young male travelers prefer?"))
print(classify_hotel_intent("Tell me about the Marriott Downtown"))
print(classify_hotel_intent("Hello, can you help me?"))
print(classify_hotel_intent("What cities have the most hotels?"))
print(classify_hotel_intent("I'm a 45 year old business traveler, suggest hotels in London"))
print(classify_hotel_intent("Find 5-star hotels in New York for families"))
print(classify_hotel_intent("Hotels with best staff ratings in Los Angeles"))
print(classify_hotel_intent("Compare Hilton vs Marriott in Paris"))

HOTEL_SEARCH
HOTEL_RECOMMENDATION
VISA_INQUIRY
REVIEW_QUERY
HOTEL_COMPARISON
RATING_ANALYSIS
HOTEL_SEARCH
TRAVELER_PREFERENCE_QUERY
HOTEL_SEARCH
HOTEL_SEARCH
STATISTICAL_QUERY
HOTEL_RECOMMENDATION
HOTEL_SEARCH
HOTEL_COMPARISON
HOTEL_COMPARISON


Rule Based

In [9]:
class HotelIntentClassifier:
    """
    Classify user intent for hotel travel queries using rule-based approach.
    Integrated with entity extraction.
    """

    def __init__(self, neo4j_driver=None):
        """
        Initialize intent classifier.

        Args:
            neo4j_driver: Optional Neo4j driver for entity extraction
        """
        self.entity_extractor = EntityExtractor(neo4j_driver)

        # Define intent patterns with keywords and weights
        self.intent_patterns = {
            Intent.HOTEL_SEARCH: {
                'keywords': ['find', 'search', 'show', 'list', 'hotels in', 'hotels near', 'available'],
            },
            Intent.HOTEL_RECOMMENDATION: {
                'keywords': ['recommend', 'suggest', 'best hotel', 'good hotel', 'looking for'],
            },
            Intent.REVIEW_QUERY: {
                'keywords': ['review', 'rating', 'what do people say', 'opinion', 'feedback', 'comment'],
            },
            Intent.HOTEL_COMPARISON: {
                'keywords': ['compare', 'versus', 'vs', 'difference', 'between', 'better'],
            },
            Intent.VISA_INQUIRY: {
                'keywords': ['visa', 'visa requirement', 'need visa', 'travel from', 'passport'],
            },
            Intent.LOCATION_BASED_QUERY: {
                'keywords': ['cities', 'countries', 'destinations', 'where', 'location'],
            },
            Intent.TRAVELER_PREFERENCE_QUERY: {
                'keywords': ['travelers prefer', 'popular with', 'suited for', 'type of traveler'],
            },
            Intent.RATING_ANALYSIS: {
                'keywords': ['best rated', 'highest rating', 'top rated', 'rating for'],
            },
            Intent.STATISTICAL_QUERY: {
                'keywords': ['average', 'how many', 'total', 'count', 'statistics', 'distribution'],
            },
            Intent.DEMOGRAPHIC_RECOMMENDATION: {
                'keywords': ['year old', 'years old', 'age', 'male', 'female', 'for me'],
            },
        }

    def preprocess_text(self, text: str) -> str:
        """
        Preprocess text for intent classification.

        Args:
            text: Raw user query

        Returns:
            Preprocessed text
        """
        # Convert to lowercase and strip whitespace
        text = text.lower().strip()

        # Remove extra whitespace
        text = ' '.join(text.split())

        return text

    def classify(self, text: str) -> Tuple[Intent, float, Dict]:
        """
        Classify user intent and extract entities.

        Args:
            text: User query text

        Returns:
            Tuple of (intent, confidence, metadata)
            - intent: Intent enum value
            - confidence: Confidence score (0-1)
            - metadata: Dictionary containing entities and other info
        """
        # Preprocess text
        preprocessed = self.preprocess_text(text)

        # Extract entities
        entities = self.entity_extractor.extract_entities(text)

        # Calculate intent scores
        intent_scores = {}

        for intent, pattern in self.intent_patterns.items():
            score = 0
            keyword_matches = 0

            for keyword in pattern['keywords']:
                if keyword in preprocessed:
                    keyword_matches += 1

            if keyword_matches > 0:
                # Base score from keyword matches
                score = min(keyword_matches * 0.2, 0.7)

                # Boost score based on entities (entity presence indicates relevance)
                entity_boost = self._calculate_entity_boost(intent, entities)
                score = min(1.0, score + entity_boost)

                intent_scores[intent] = score

        # Special case: demographic recommendation if age/gender found
        if ('age' in entities or 'gender' in entities) and 'traveler_type' in entities:
            if Intent.DEMOGRAPHIC_RECOMMENDATION not in intent_scores or \
               intent_scores.get(Intent.DEMOGRAPHIC_RECOMMENDATION, 0) < 0.8:
                intent_scores[Intent.DEMOGRAPHIC_RECOMMENDATION] = 0.80

        # Special case: visa inquiry if multiple countries mentioned
        if len(entities.get('country', [])) >= 2:
            if Intent.VISA_INQUIRY not in intent_scores or \
               intent_scores.get(Intent.VISA_INQUIRY, 0) < 0.7:
                intent_scores[Intent.VISA_INQUIRY] = 0.70

        # Get best intent
        if intent_scores:
            best_intent = max(intent_scores.items(), key=lambda x: x[1])
            intent = best_intent[0]
            confidence = best_intent[1]
        else:
            intent = Intent.UNKNOWN
            confidence = 0.0

        # Prepare metadata
        metadata = {
            'entities': entities,
            'preprocessed_text': preprocessed,
            'original_text': text,
            'all_intent_scores': dict(intent_scores)
        }

        return intent, confidence, metadata

    def _calculate_entity_boost(self, intent: Intent, entities: Dict) -> float:
        """
        Calculate confidence boost based on entity presence for specific intents.

        Args:
            intent: Current intent being evaluated
            entities: Extracted entities

        Returns:
            Boost value (0-0.3)
        """
        boost = 0.0

        # Different intents benefit from different entities
        if intent == Intent.HOTEL_SEARCH:
            if 'city' in entities or 'country' in entities:
                boost += 0.15
            if 'star_rating' in entities or 'feature' in entities:
                boost += 0.10

        elif intent == Intent.HOTEL_RECOMMENDATION:
            if 'traveler_type' in entities:
                boost += 0.15
            if 'city' in entities:
                boost += 0.10

        elif intent == Intent.REVIEW_QUERY:
            if 'hotel' in entities or 'city' in entities:
                boost += 0.15

        elif intent == Intent.VISA_INQUIRY:
            if 'country' in entities and len(entities['country']) >= 1:
                boost += 0.20

        elif intent == Intent.DEMOGRAPHIC_RECOMMENDATION:
            if 'age' in entities or 'gender' in entities:
                boost += 0.15
            if 'traveler_type' in entities:
                boost += 0.10

        return min(0.3, boost)  # Cap at 0.3

Testing

In [10]:
# Test entity extraction independently
print("=" * 80)
print("ENTITY EXTRACTION TESTS")
print("=" * 80)

extractor = EntityExtractor()

test_queries_entities = [
    "Find 5-star hotels in New York",
    "I'm a 28 year old male business traveler",
    "Hotels with good cleanliness in Los Angeles",
    "Compare hotels in United States vs United Kingdom",
    "Do I need a visa to travel from Egypt to France?"
]

for query in test_queries_entities:
    entities = extractor.extract_entities(query)
    print(f"\nQuery: {query}")
    print(f"Entities: {entities}")
    print("-" * 80)

ENTITY EXTRACTION TESTS

Query: Find 5-star hotels in New York
Entities: {'city': ['new york'], 'star_rating': ['5']}
--------------------------------------------------------------------------------

Query: I'm a 28 year old male business traveler
Entities: {'traveler_type': ['business', 'business traveler'], 'age': ['28'], 'gender': ['male']}
--------------------------------------------------------------------------------

Query: Hotels with good cleanliness in Los Angeles
Entities: {'feature': ['cleanliness']}
--------------------------------------------------------------------------------

Query: Compare hotels in United States vs United Kingdom
Entities: {'country': ['united states', 'united kingdom']}
--------------------------------------------------------------------------------

Query: Do I need a visa to travel from Egypt to France?
Entities: {'country': ['egypt', 'france']}
--------------------------------------------------------------------------------


In [11]:
# Test complete intent classification pipeline
print("\n" + "=" * 80)
print("INTENT CLASSIFICATION + ENTITY EXTRACTION - INTEGRATED TESTS")
print("=" * 80)

classifier = HotelIntentClassifier()

test_queries = [
    "Find hotels in Paris",
    "Recommend a hotel for a 28-year-old female solo traveler",
    "Do I need a visa to travel from Egypt to France?",
    "What do people say about the Hilton in Cairo?",
    "Compare hotels in Rome vs Florence",
    "What's the average rating of 5-star hotels?",
    "Show me hotels with good cleanliness ratings in Dubai",
    "Which hotels do young male travelers prefer?",
    "Tell me about the Marriott Downtown",
    "Hello, can you help me?",
    "What cities have the most hotels?",
    "I'm a 45 year old business traveler, suggest hotels in London",
    "Find 5-star hotels in New York for families",
    "Hotels with best staff ratings in Los Angeles",
    "Compare Hilton vs Marriott in Paris"
]

for query in test_queries:
    intent, confidence, metadata = classifier.classify(query)

    print(f"\nQuery: {query}")
    print(f"Intent: {intent.value}")
    print(f"Confidence: {confidence:.2f}")
    print(f"Entities: {metadata['entities']}")

    # Show top 3 intent scores for transparency
    all_scores = metadata['all_intent_scores']
    if all_scores:
        top_3 = sorted(all_scores.items(), key=lambda x: x[1], reverse=True)[:3]
        print(f"Top 3 intents: {[(i.value, f'{s:.2f}') for i, s in top_3]}")

    print("-" * 80)


INTENT CLASSIFICATION + ENTITY EXTRACTION - INTEGRATED TESTS

Query: Find hotels in Paris
Intent: HOTEL_SEARCH
Confidence: 0.55
Entities: {'city': ['paris']}
Top 3 intents: [('HOTEL_SEARCH', '0.55')]
--------------------------------------------------------------------------------

Query: Recommend a hotel for a 28-year-old female solo traveler
Intent: DEMOGRAPHIC_RECOMMENDATION
Confidence: 0.80
Entities: {'traveler_type': ['solo', 'solo traveler'], 'gender': ['female']}
Top 3 intents: [('DEMOGRAPHIC_RECOMMENDATION', '0.80'), ('HOTEL_RECOMMENDATION', '0.35')]
--------------------------------------------------------------------------------

Query: Do I need a visa to travel from Egypt to France?
Intent: VISA_INQUIRY
Confidence: 0.70
Entities: {'country': ['egypt', 'france']}
Top 3 intents: [('VISA_INQUIRY', '0.70')]
--------------------------------------------------------------------------------

Query: What do people say about the Hilton in Cairo?
Intent: REVIEW_QUERY
Confidence: 0.35


USER INPUT EMBEDDING

In [12]:
# Model 1: SBERT (small, fast)
sbert_model = SentenceTransformer('all-MiniLM-L6-v2')

# Model 2: BGE (larger, better quality)
bge_model = SentenceTransformer('BAAI/bge-large-en-v1.5')

In [13]:
def embed_query(query_text):
    """
    Embed the same query using both SBERT and BGE.

    Args:
        query_text: User query string

    Returns:
        Dictionary with embeddings from both models
    """

    # SBERT embedding (384 dims)
    sbert_embedding = sbert_model.encode(query_text)

    # BGE embedding (1024 dims)
    bge_embedding = bge_model.encode(query_text)

    return {
        'sbert': sbert_embedding,
        'bge': bge_embedding
    }

In [14]:
query = "Find 5-star hotels in Paris with excellent cleanliness"

embeddings = embed_query(query)

print(f"Query: {query}\n")

print(f"SBERT embedding shape: {embeddings['sbert'].shape}")
print(f"SBERT (first 5): {embeddings['sbert'][:5]}\n")

print(f"BGE embedding shape: {embeddings['bge'].shape}")
print(f"BGE (first 5): {embeddings['bge'][:5]}")

Query: Find 5-star hotels in Paris with excellent cleanliness

SBERT embedding shape: (384,)
SBERT (first 5): [ 0.07261952 -0.04525769  0.07977294  0.00909198  0.04659884]

BGE embedding shape: (1024,)
BGE (first 5): [-0.0063732   0.02636965  0.01333166  0.04589348 -0.01661471]


2. GRAPH RETRIEVAL LAYER

BASELINE

In [15]:
class QueryMapper:
    """Maps extracted entities to query parameters for different intents."""
    
    def map_entities_to_parameters(self, intent: str, entities: dict) -> dict:
        """
        Convert extracted entities into query parameters for the selected intent.
        
        Args:
            intent: The classified intent (e.g., 'HOTEL_SEARCH_BASIC')
            entities: Dictionary of extracted entities
            
        Returns:
            Dictionary of parameters for the Cypher query
        """
        params = {}
        
        # Common mappings used across multiple intents
        if 'city' in entities and entities['city']:
            params['city'] = entities['city'][0]  # Take first match
        
        if 'country' in entities and entities['country']:
            params['country'] = entities['country'][0]
        
        if 'hotel' in entities and entities['hotel']:
            params['hotel_name'] = entities['hotel'][0]
        
        # Intent-specific mappings
        if intent == 'HOTEL_SEARCH_BASIC':
            if 'star_rating' in entities and entities['star_rating']:
                params['min_star_rating'] = int(entities['star_rating'][0])
            else:
                params['min_star_rating'] = None
        
        elif intent == 'HOTEL_SEARCH_FEATURE':
            if 'feature' in entities and entities['feature']:
                params['feature_type'] = entities['feature'][0]
                params['min_score'] = 4.0  # Default threshold for "good" features
            else:
                params['feature_type'] = None
                params['min_score'] = 3.5  # Lower default if no feature specified
        
        elif intent == 'HOTEL_SEARCH_DEMOGRAPHIC':
            if 'traveler_type' in entities and entities['traveler_type']:
                params['traveler_type'] = entities['traveler_type'][0].capitalize()
            else:
                params['traveler_type'] = None
            
            if 'age' in entities and entities['age']:
                age = int(entities['age'][0])
                params['age_min'] = age - 5
                params['age_max'] = age + 5
            else:
                params['age_min'] = None
                params['age_max'] = None
            
            if 'gender' in entities and entities['gender']:
                params['gender'] = entities['gender'][0].capitalize()
            else:
                params['gender'] = None
        
        elif intent == 'REVIEW_QUERY_BASIC':
            params['limit'] = 20
            
            # Check for sentiment indicators (positive/negative)
            # You could enhance this by detecting "negative", "positive", "bad", "good" in query
            params['min_score'] = None
            params['max_score'] = None
            
            # If you detect "negative" in the original query, you could set:
            # params['max_score'] = 3
        
        elif intent == 'REVIEW_QUERY_DEMOGRAPHIC':
            params['limit'] = 20
            
            if 'gender' in entities and entities['gender']:
                params['gender'] = entities['gender'][0].capitalize()
            else:
                params['gender'] = None
            
            if 'age' in entities and entities['age']:
                age = int(entities['age'][0])
                params['age_min'] = age - 5
                params['age_max'] = age + 5
            else:
                params['age_min'] = None
                params['age_max'] = None
            
            if 'traveler_type' in entities and entities['traveler_type']:
                params['traveler_type'] = entities['traveler_type'][0].capitalize()
            else:
                params['traveler_type'] = None
        
        elif intent == 'HOTEL_RECOMMENDATION':
            if 'feature' in entities and entities['feature']:
                params['feature_preference'] = entities['feature'][0]
                params['min_score'] = 4.0
            else:
                params['feature_preference'] = None
                params['min_score'] = 3.5
            
            if 'traveler_type' in entities and entities['traveler_type']:
                params['traveler_type'] = entities['traveler_type'][0].capitalize()
            else:
                params['traveler_type'] = None
        
        elif intent == 'DEMOGRAPHIC_RECOMMENDATION':
            if 'age' in entities and entities['age']:
                params['age'] = int(entities['age'][0])
            else:
                params['age'] = 30  # Default age if not specified
            
            if 'gender' in entities and entities['gender']:
                params['gender'] = entities['gender'][0].capitalize()
            else:
                params['gender'] = None
            
            if 'traveler_type' in entities and entities['traveler_type']:
                params['traveler_type'] = entities['traveler_type'][0].capitalize()
            else:
                params['traveler_type'] = None
        
        elif intent == 'LOCATION_QUERY_STATS':
            if 'star_rating' in entities and entities['star_rating']:
                params['min_star_rating'] = int(entities['star_rating'][0])
            else:
                params['min_star_rating'] = None
        
        elif intent == 'LOCATION_QUERY_DEMOGRAPHIC':
            if 'gender' in entities and entities['gender']:
                params['gender'] = entities['gender'][0].capitalize()
            else:
                params['gender'] = None
            
            if 'age' in entities and entities['age']:
                age = int(entities['age'][0])
                params['age_min'] = age - 5
                params['age_max'] = age + 5
            else:
                params['age_min'] = None
                params['age_max'] = None
            
            if 'traveler_type' in entities and entities['traveler_type']:
                params['traveler_type'] = entities['traveler_type'][0].capitalize()
            else:
                params['traveler_type'] = None
        
        elif intent == 'VISA_INQUIRY':
            if 'country' in entities and len(entities['country']) >= 2:
                params['from_country'] = entities['country'][0]
                params['to_country'] = entities['country'][1]
            elif 'country' in entities and len(entities['country']) == 1:
                # If only one country mentioned, you might need to prompt user
                params['from_country'] = entities['country'][0]
                params['to_country'] = None
            else:
                params['from_country'] = None
                params['to_country'] = None
        
        elif intent == 'COMPARISON':
            if 'hotel' in entities and len(entities['hotel']) >= 2:
                params['hotel_name_1'] = entities['hotel'][0]
                params['hotel_name_2'] = entities['hotel'][1]
                params['city_1'] = None
                params['city_2'] = None
            elif 'city' in entities and len(entities['city']) >= 2:
                params['hotel_name_1'] = None
                params['hotel_name_2'] = None
                params['city_1'] = entities['city'][0]
                params['city_2'] = entities['city'][1]
            elif 'hotel' in entities and len(entities['hotel']) == 1:
                params['hotel_name_1'] = entities['hotel'][0]
                params['hotel_name_2'] = None
                params['city_1'] = None
                params['city_2'] = None
            else:
                params['hotel_name_1'] = None
                params['hotel_name_2'] = None
                params['city_1'] = None
                params['city_2'] = None
        
        elif intent == 'STATISTICAL_QUERY':
            if 'star_rating' in entities and entities['star_rating']:
                params['star_rating'] = int(entities['star_rating'][0])
            else:
                params['star_rating'] = None
            
            params['metric_type'] = 'all'  # Could be enhanced to detect specific metrics
        
        return params


In [16]:
class QueryExecutor:
    """Executes Cypher queries against Neo4j database."""
    
    def __init__(self, uri: str, user: str, password: str):
        """
        Initialize the query executor with Neo4j connection.
        
        Args:
            uri: Neo4j connection URI (e.g., 'bolt://localhost:7687')
            user: Database username
            password: Database password
        """
        self.driver = GraphDatabase.driver(uri, auth=(user, password))
        self.query_templates = self._load_query_templates()
    
    def _load_query_templates(self) -> Dict[str, str]:
        """Load all 12 Cypher query templates."""
        return {
            # Query 1: Basic hotel search by location and rating
            'HOTEL_SEARCH_BASIC': """
                MATCH (h:Hotel)-[:LOCATED_IN]->(c:City)-[:LOCATED_IN]->(co:Country)
                WHERE (toLower(c.name) = toLower($city) OR $city IS NULL)
                  AND (toLower(co.name) = toLower($country) OR $country IS NULL)
                  AND (h.star_rating >= $min_star_rating OR $min_star_rating IS NULL)
                RETURN h.name AS hotel_name, 
                       h.star_rating AS star_rating,
                       h.average_reviews_score AS avg_score,
                       c.name AS city,
                       co.name AS country
                ORDER BY h.average_reviews_score DESC
                LIMIT 10
            """,
            
            # Query 2: Feature-based hotel search
            'HOTEL_SEARCH_FEATURE': """
                MATCH (h:Hotel)-[:LOCATED_IN]->(c:City)
                WHERE (toLower(c.name) = toLower($city) OR $city IS NULL)
                  AND (
                    ($feature_type = 'cleanliness' AND h.cleanliness_base >= $min_score) OR
                    ($feature_type = 'comfort' AND h.comfort_base >= $min_score) OR
                    ($feature_type = 'facilities' AND h.facilities_base >= $min_score) OR
                    ($feature_type IS NULL AND h.average_reviews_score >= $min_score)
                  )
                RETURN h.name AS hotel_name,
                       h.cleanliness_base AS cleanliness,
                       h.comfort_base AS comfort,
                       h.facilities_base AS facilities,
                       h.average_reviews_score AS overall_score,
                       c.name AS city
                ORDER BY 
                  CASE $feature_type
                    WHEN 'cleanliness' THEN h.cleanliness_base
                    WHEN 'comfort' THEN h.comfort_base
                    WHEN 'facilities' THEN h.facilities_base
                    ELSE h.average_reviews_score
                  END DESC
                LIMIT 10
            """,
            
            # Query 3: Demographic-filtered hotel search
            'HOTEL_SEARCH_DEMOGRAPHIC': """
                MATCH (t:Traveller)-[:STAYED_AT]->(h:Hotel)-[:LOCATED_IN]->(c:City)
                WHERE (toLower(c.name) = toLower($city) OR $city IS NULL)
                  AND (toLower(t.type) = toLower($traveler_type) OR $traveler_type IS NULL)
                  AND (t.age >= $age_min OR $age_min IS NULL)
                  AND (t.age <= $age_max OR $age_max IS NULL)
                  AND (toLower(t.gender) = toLower($gender) OR $gender IS NULL)
                WITH h, c, COUNT(DISTINCT t) AS traveler_count
                WHERE traveler_count >= 3
                RETURN h.name AS hotel_name,
                       h.star_rating AS star_rating,
                       h.average_reviews_score AS avg_score,
                       c.name AS city,
                       traveler_count AS matching_travelers
                ORDER BY traveler_count DESC, h.average_reviews_score DESC
                LIMIT 10
            """,
            
            # Query 4: Basic review retrieval
            'REVIEW_QUERY_BASIC': """
                MATCH (r:Review)-[:REVIEWED]->(h:Hotel)-[:LOCATED_IN]->(c:City)
                MATCH (t:Traveller)-[:WROTE]->(r)
                WHERE (toLower(c.name) = toLower($city) OR $city IS NULL)
                  AND (toLower(h.name) CONTAINS toLower($hotel_name) OR $hotel_name IS NULL)
                  AND (r.score_overall >= $min_score OR $min_score IS NULL)
                  AND (r.score_overall <= $max_score OR $max_score IS NULL)
                RETURN h.name AS hotel_name,
                       c.name AS city,
                       r.text AS review_text,
                       r.score_overall AS overall_score,
                       r.score_cleanliness AS cleanliness_score,
                       r.score_comfort AS comfort_score,
                       r.date AS review_date,
                       t.type AS traveler_type,
                       t.age AS traveler_age,
                       t.gender AS traveler_gender
                ORDER BY r.date DESC
                LIMIT $limit
            """,
            
            # Query 5: Demographic-filtered review retrieval
            'REVIEW_QUERY_DEMOGRAPHIC': """
                MATCH (t:Traveller)-[:WROTE]->(r:Review)-[:REVIEWED]->(h:Hotel)
                WHERE (toLower(h.name) CONTAINS toLower($hotel_name) OR $hotel_name IS NULL)
                  AND (toLower(t.gender) = toLower($gender) OR $gender IS NULL)
                  AND (t.age >= $age_min OR $age_min IS NULL)
                  AND (t.age <= $age_max OR $age_max IS NULL)
                  AND (toLower(t.type) = toLower($traveler_type) OR $traveler_type IS NULL)
                RETURN h.name AS hotel_name,
                       r.text AS review_text,
                       r.score_overall AS overall_score,
                       r.date AS review_date,
                       t.gender AS reviewer_gender,
                       t.age AS reviewer_age,
                       t.type AS reviewer_type
                ORDER BY r.date DESC
                LIMIT $limit
            """,
            
            # Query 6: Feature-based hotel recommendation
            'HOTEL_RECOMMENDATION': """
                MATCH (h:Hotel)-[:LOCATED_IN]->(c:City)
                WHERE (toLower(c.name) = toLower($city) OR $city IS NULL)
                OPTIONAL MATCH (t:Traveller)-[:STAYED_AT]->(h)
                WHERE (toLower(t.type) = toLower($traveler_type) OR $traveler_type IS NULL)
                WITH h, c, COUNT(DISTINCT t) AS matching_travelers,
                  CASE $feature_preference
                    WHEN 'cleanliness' THEN h.cleanliness_base
                    WHEN 'comfort' THEN h.comfort_base
                    WHEN 'facilities' THEN h.facilities_base
                    WHEN 'staff' THEN h.average_reviews_score
                    WHEN 'value' THEN h.average_reviews_score
                    ELSE h.average_reviews_score
                  END AS feature_score
                WHERE feature_score >= $min_score OR $min_score IS NULL
                RETURN h.name AS hotel_name,
                       h.star_rating AS star_rating,
                       h.cleanliness_base AS cleanliness,
                       h.comfort_base AS comfort,
                       h.facilities_base AS facilities,
                       h.average_reviews_score AS overall_score,
                       c.name AS city,
                       matching_travelers AS similar_travelers
                ORDER BY feature_score DESC, matching_travelers DESC
                LIMIT 10
            """,
            
            # Query 7: Demographic-based recommendation
            'DEMOGRAPHIC_RECOMMENDATION': """
                MATCH (similar:Traveller)-[:STAYED_AT]->(h:Hotel)-[:LOCATED_IN]->(c:City)
                WHERE (similar.age >= $age - 5 AND similar.age <= $age + 5)
                  AND (toLower(similar.gender) = toLower($gender) OR $gender IS NULL)
                  AND (toLower(similar.type) = toLower($traveler_type) OR $traveler_type IS NULL)
                  AND (toLower(c.name) = toLower($city) OR $city IS NULL)
                OPTIONAL MATCH (similar)-[:WROTE]->(r:Review)-[:REVIEWED]->(h)
                WITH h, c, 
                     COUNT(DISTINCT similar) AS similar_traveler_count,
                     AVG(r.score_overall) AS avg_rating_from_similar
                WHERE similar_traveler_count >= 2
                RETURN h.name AS hotel_name,
                       h.star_rating AS star_rating,
                       h.average_reviews_score AS overall_avg_score,
                       c.name AS city,
                       similar_traveler_count AS travelers_like_you,
                       COALESCE(avg_rating_from_similar, h.average_reviews_score) AS score_from_similar_travelers
                ORDER BY similar_traveler_count DESC, score_from_similar_travelers DESC
                LIMIT 10
            """,
            
            # Query 8: Location statistics
            'LOCATION_QUERY_STATS': """
                MATCH (h:Hotel)-[:LOCATED_IN]->(c:City)-[:LOCATED_IN]->(co:Country)
                WHERE (toLower(co.name) = toLower($country) OR $country IS NULL)
                  AND (h.star_rating >= $min_star_rating OR $min_star_rating IS NULL)
                WITH c, co, COUNT(h) AS hotel_count, AVG(h.average_reviews_score) AS avg_rating
                WHERE hotel_count > 0
                RETURN c.name AS city,
                       co.name AS country,
                       hotel_count AS number_of_hotels,
                       ROUND(avg_rating * 100) / 100 AS average_rating
                ORDER BY hotel_count DESC, avg_rating DESC
                LIMIT 20
            """,
            
            # Query 9: Location demographic patterns
            'LOCATION_QUERY_DEMOGRAPHIC': """
                MATCH (t:Traveller)-[:STAYED_AT]->(h:Hotel)-[:LOCATED_IN]->(c:City)-[:LOCATED_IN]->(co:Country)
                WHERE (toLower(t.gender) = toLower($gender) OR $gender IS NULL)
                  AND (t.age >= $age_min OR $age_min IS NULL)
                  AND (t.age <= $age_max OR $age_max IS NULL)
                  AND (toLower(t.type) = toLower($traveler_type) OR $traveler_type IS NULL)
                WITH c, co, COUNT(DISTINCT t) AS traveler_count
                WHERE traveler_count >= 3
                RETURN c.name AS city,
                       co.name AS country,
                       traveler_count AS matching_travelers
                ORDER BY traveler_count DESC
                LIMIT 15
            """,
            
            # Query 10: Visa inquiry
            'VISA_INQUIRY': """
                MATCH (from:Country)-[v:NEEDS_VISA]->(to:Country)
                WHERE toLower(from.name) = toLower($from_country)
                  AND toLower(to.name) = toLower($to_country)
                RETURN from.name AS from_country,
                       to.name AS to_country,
                       v.visa_type AS visa_type,
                       'Yes' AS visa_required
                UNION
                MATCH (from:Country), (to:Country)
                WHERE toLower(from.name) = toLower($from_country)
                  AND toLower(to.name) = toLower($to_country)
                  AND NOT EXISTS((from)-[:NEEDS_VISA]->(to))
                RETURN from.name AS from_country,
                       to.name AS to_country,
                       NULL AS visa_type,
                       'No' AS visa_required
            """,
            
            # Query 11: Hotel comparison
            'COMPARISON': """
                MATCH (h1:Hotel)
                WHERE toLower(h1.name) CONTAINS toLower($hotel_name_1)
                  OR (EXISTS((h1)-[:LOCATED_IN]->(:City {name: $city_1})) AND $hotel_name_1 IS NULL)
                OPTIONAL MATCH (h1)-[:LOCATED_IN]->(c1:City)
                WITH h1, c1
                LIMIT 1
                
                MATCH (h2:Hotel)
                WHERE toLower(h2.name) CONTAINS toLower($hotel_name_2)
                  OR (EXISTS((h2)-[:LOCATED_IN]->(:City {name: $city_2})) AND $hotel_name_2 IS NULL)
                OPTIONAL MATCH (h2)-[:LOCATED_IN]->(c2:City)
                WITH h1, c1, h2, c2
                LIMIT 1
                
                RETURN h1.name AS hotel_1,
                       c1.name AS city_1,
                       h1.star_rating AS hotel_1_stars,
                       h1.average_reviews_score AS hotel_1_rating,
                       h1.cleanliness_base AS hotel_1_cleanliness,
                       h1.comfort_base AS hotel_1_comfort,
                       h1.facilities_base AS hotel_1_facilities,
                       h2.name AS hotel_2,
                       c2.name AS city_2,
                       h2.star_rating AS hotel_2_stars,
                       h2.average_reviews_score AS hotel_2_rating,
                       h2.cleanliness_base AS hotel_2_cleanliness,
                       h2.comfort_base AS hotel_2_comfort,
                       h2.facilities_base AS hotel_2_facilities
            """,
            
            # Query 12: Statistical aggregations
            'STATISTICAL_QUERY': """
                MATCH (h:Hotel)-[:LOCATED_IN]->(c:City)-[:LOCATED_IN]->(co:Country)
                WHERE (toLower(c.name) = toLower($city) OR $city IS NULL)
                  AND (toLower(co.name) = toLower($country) OR $country IS NULL)
                  AND (h.star_rating = $star_rating OR $star_rating IS NULL)
                OPTIONAL MATCH (t:Traveller)-[:STAYED_AT]->(h)
                OPTIONAL MATCH (r:Review)-[:REVIEWED]->(h)
                WITH c, co, h,
                     COUNT(DISTINCT r) AS review_count,
                     AVG(h.average_reviews_score) AS avg_overall_rating,
                     AVG(h.cleanliness_base) AS avg_cleanliness,
                     AVG(h.comfort_base) AS avg_comfort,
                     AVG(h.facilities_base) AS avg_facilities,
                     COUNT(DISTINCT t) AS traveler_count
                RETURN c.name AS city,
                       co.name AS country,
                       COUNT(h) AS total_hotels,
                       ROUND(avg_overall_rating * 100) / 100 AS average_rating,
                       ROUND(avg_cleanliness * 100) / 100 AS average_cleanliness,
                       ROUND(avg_comfort * 100) / 100 AS average_comfort,
                       ROUND(avg_facilities * 100) / 100 AS average_facilities,
                       SUM(review_count) AS total_reviews,
                       SUM(traveler_count) AS total_travelers
            """
        }
    
    def execute_query(self, intent: str, params: Dict[str, Any]) -> List[Dict]:
        """
        Execute the appropriate query with parameters.
        
        Args:
            intent: The intent type (matches query template key)
            params: Dictionary of parameters for the query
            
        Returns:
            List of result dictionaries
        """
        query = self.query_templates.get(intent)
        if not query:
            raise ValueError(f"Unknown intent: {intent}. Available intents: {list(self.query_templates.keys())}")
        
        # Ensure all possible parameters have a value (None if not provided)
        all_possible_params = [
            'city', 'country', 'hotel_name', 'min_star_rating', 'star_rating',
            'feature_type', 'min_score', 'feature_preference',
            'traveler_type', 'age', 'age_min', 'age_max', 'gender',
            'limit', 'min_score', 'max_score',
            'from_country', 'to_country',
            'hotel_name_1', 'hotel_name_2', 'city_1', 'city_2',
            'metric_type'
        ]
        
        for key in all_possible_params:
            if key not in params:
                params[key] = None
        
        try:
            with self.driver.session() as session:
                result = session.run(query, **params)
                return [dict(record) for record in result]
        except Exception as e:
            print(f"Error executing query for intent '{intent}': {str(e)}")
            print(f"Parameters: {params}")
            raise
    
    def close(self):
        """Close the database connection."""
        self.driver.close()


In [18]:
# Initialize components
mapper = QueryMapper()
executor = QueryExecutor(
    uri="neo4j://127.0.0.1:7687",
    user="neo4j",
    password="sSNrBXVgeDYG3y9Jv2pyLX_PgKolKhpv4ePEYChwHlw"
)

try:
    # Example 1: Basic hotel search
    entities_1 = {
        'city': ['Paris'],
        'star_rating': ['5']
    }
    intent_1 = 'HOTEL_SEARCH_BASIC'
    params_1 = mapper.map_entities_to_parameters(intent_1, entities_1)
    results_1 = executor.execute_query(intent_1, params_1)
    print(f"Found {len(results_1)} hotels")
    print(results_1)
    
    # Example 2: Demographic recommendation
    entities_2 = {
        'age': ['28'],
        'gender': ['female'],
        'traveler_type': ['solo'],
        'city': ['Paris']
    }
    intent_2 = 'DEMOGRAPHIC_RECOMMENDATION'
    params_2 = mapper.map_entities_to_parameters(intent_2, entities_2)
    results_2 = executor.execute_query(intent_2, params_2)
    print(f"Found {len(results_2)} recommendations")
    print(results_2)
    
    # Example 3: Visa inquiry
    entities_3 = {
        'country': ['Egypt', 'France']
    }
    intent_3 = 'VISA_INQUIRY'
    params_3 = mapper.map_entities_to_parameters(intent_3, entities_3)
    results_3 = executor.execute_query(intent_3, params_3)
    print(f"Visa info: {results_3}")
    
finally:
    executor.close()

Found 1 hotels
[{'hotel_name': "L'Étoile Palace", 'star_rating': 5.0, 'avg_score': 8.9470647265429, 'city': 'Paris', 'country': 'France'}]
Found 0 recommendations
[]
Visa info: [{'from_country': 'Egypt', 'to_country': 'France', 'visa_type': 'Tourist Visa', 'visa_required': 'Yes'}]


EMBEDDINGS

In [ ]:
class EnhancedFeatureVectorEmbedder:
    """
    Create feature vector embeddings for:
    1. Hotels (with review text AND scores)
    2. Visa requirements (country pairs with visa info)
    """
    
    def __init__(self, neo4j_uri: str, neo4j_user: str, neo4j_password: str):
        """Initialize with Neo4j connection and embedding models."""
        # Connect to Neo4j
        self.driver = GraphDatabase.driver(neo4j_uri, auth=(neo4j_user, neo4j_password))
        
        # Load both embedding models
        print("Loading SBERT model...")
        self.sbert_model = SentenceTransformer('all-MiniLM-L6-v2')
        
        print("Loading BGE model...")
        self.bge_model = SentenceTransformer('BAAI/bge-large-en-v1.5')
        
        print("Models loaded successfully!")
    
    # ========================================================================
    # HOTEL EMBEDDINGS (Enhanced with Review Scores)
    # ========================================================================
    
    def fetch_hotels_with_features(self) -> List[Dict[str, Any]]:
        """
        Fetch all hotels with their properties, reviews, AND review scores.
        """
        query = """
        MATCH (h:Hotel)-[:LOCATED_IN]->(c:City)-[:LOCATED_IN]->(co:Country)
        OPTIONAL MATCH (r:Review)-[:REVIEWED]->(h)
        OPTIONAL MATCH (t:Traveller)-[:STAYED_AT]->(h)
        
        WITH h, c, co,
             COLLECT(DISTINCT r.text)[0..5] AS sample_reviews,
             COLLECT(DISTINCT {
                 overall: r.score_overall,
                 cleanliness: r.score_cleanliness,
                 comfort: r.score_comfort,
                 facilities: r.score_facilities,
                 location: r.score_location,
                 staff: r.score_staff,
                 value_for_money: r.score_value_for_money
             })[0..5] AS review_scores,
             COUNT(DISTINCT r) AS review_count,
             COUNT(DISTINCT t) AS traveler_count,
             COLLECT(DISTINCT t.type) AS traveler_types
        
        RETURN h.hotel_id AS hotel_id,
               h.name AS hotel_name,
               h.star_rating AS star_rating,
               h.average_reviews_score AS avg_score,
               h.cleanliness_base AS cleanliness,
               h.comfort_base AS comfort,
               h.facilities_base AS facilities,
               c.name AS city,
               co.name AS country,
               sample_reviews,
               review_scores,
               review_count,
               traveler_count,
               traveler_types
        """
        
        with self.driver.session() as session:
            result = session.run(query)
            hotels = [dict(record) for record in result]
        
        print(f"Fetched {len(hotels)} hotels from Neo4j")
        return hotels
    
    def create_hotel_feature_text(self, hotel: Dict[str, Any]) -> str:
        """
        Combine hotel properties into rich text representation.
        NOW INCLUDES: Review scores alongside review text!
        
        Args:
            hotel: Dictionary containing hotel properties
            
        Returns:
            Combined text representation
        """
        parts = []
        
        # Basic info
        parts.append(f"Hotel: {hotel['hotel_name']}")
        parts.append(f"Location: {hotel['city']}, {hotel['country']}")
        parts.append(f"Star rating: {hotel['star_rating']} stars")
        
        # Quality scores (hotel base scores)
        if hotel['avg_score']:
            parts.append(f"Overall rating: {hotel['avg_score']:.1f}/5")
        if hotel['cleanliness']:
            parts.append(f"Cleanliness: {hotel['cleanliness']:.1f}/5")
        if hotel['comfort']:
            parts.append(f"Comfort: {hotel['comfort']:.1f}/5")
        if hotel['facilities']:
            parts.append(f"Facilities: {hotel['facilities']:.1f}/5")
        
        # Traveler information
        if hotel['traveler_count'] and hotel['traveler_count'] > 0:
            parts.append(f"Popular with {hotel['traveler_count']} travelers")
        
        if hotel['traveler_types']:
            traveler_types_str = ', '.join([t for t in hotel['traveler_types'] if t])
            if traveler_types_str:
                parts.append(f"Traveler types: {traveler_types_str}")
        
        # ====================================================================
        # ENHANCED: Review scores + review text together!
        # ====================================================================
        
        if hotel['review_scores']:
            # Calculate average scores across sample reviews
            valid_scores = [s for s in hotel['review_scores'] if s and any(s.values())]
            
            if valid_scores:
                # Aggregate scores
                avg_scores = {
                    'overall': np.mean([s['overall'] for s in valid_scores if s.get('overall')]),
                    'cleanliness': np.mean([s['cleanliness'] for s in valid_scores if s.get('cleanliness')]),
                    'comfort': np.mean([s['comfort'] for s in valid_scores if s.get('comfort')]),
                    'facilities': np.mean([s['facilities'] for s in valid_scores if s.get('facilities')]),
                    'location': np.mean([s['location'] for s in valid_scores if s.get('location')]),
                    'staff': np.mean([s['staff'] for s in valid_scores if s.get('staff')]),
                    'value_for_money': np.mean([s['value_for_money'] for s in valid_scores if s.get('value_for_money')])
                }
                
                # Add detailed score breakdown
                score_parts = []
                if not np.isnan(avg_scores['overall']):
                    score_parts.append(f"guest overall: {avg_scores['overall']:.1f}/5")
                if not np.isnan(avg_scores['cleanliness']):
                    score_parts.append(f"guest cleanliness: {avg_scores['cleanliness']:.1f}/5")
                if not np.isnan(avg_scores['comfort']):
                    score_parts.append(f"guest comfort: {avg_scores['comfort']:.1f}/5")
                if not np.isnan(avg_scores['facilities']):
                    score_parts.append(f"guest facilities: {avg_scores['facilities']:.1f}/5")
                if not np.isnan(avg_scores['location']):
                    score_parts.append(f"guest location: {avg_scores['location']:.1f}/5")
                if not np.isnan(avg_scores['staff']):
                    score_parts.append(f"guest staff: {avg_scores['staff']:.1f}/5")
                if not np.isnan(avg_scores['value_for_money']):
                    score_parts.append(f"guest value: {avg_scores['value_for_money']:.1f}/5")
                
                if score_parts:
                    parts.append(f"Guest ratings: {', '.join(score_parts)}")
        
        # Sample reviews (textual content)
        if hotel['sample_reviews']:
            valid_reviews = [r for r in hotel['sample_reviews'] if r and len(r.strip()) > 10]
            if valid_reviews:
                review_text = ' '.join(valid_reviews[:3])
                parts.append(f"Guest reviews: {review_text}")
        
        # Combine all parts
        feature_text = '. '.join(parts)
        
        return feature_text
    
    # ========================================================================
    # VISA EMBEDDINGS
    # ========================================================================
    
    def fetch_visa_requirements(self) -> List[Dict[str, Any]]:
        """
        Fetch all visa requirement relationships from Neo4j.
        Each visa requirement is a (Country)-[:NEEDS_VISA]->(Country) relationship.
        
        Returns:
            List of dictionaries containing visa requirement data
        """
        query = """
        MATCH (from:Country)-[v:NEEDS_VISA]->(to:Country)
        RETURN from.name AS from_country,
               to.name AS to_country,
               v.visa_type AS visa_type
        """
        
        with self.driver.session() as session:
            result = session.run(query)
            visa_reqs = [dict(record) for record in result]
        
        print(f"Fetched {len(visa_reqs)} visa requirements from Neo4j")
        return visa_reqs
    
    def create_visa_feature_text(self, visa_req: Dict[str, Any]) -> str:
        """
        Create text representation of a visa requirement.
        
        Args:
            visa_req: Dictionary with from_country, to_country, visa_type
            
        Returns:
            Text representation of visa requirement
        """
        from_country = visa_req['from_country']
        to_country = visa_req['to_country']
        visa_type = visa_req.get('visa_type', 'required')
        
        # Create rich textual representation
        text = (
            f"Visa requirement: Travelers from {from_country} need a visa to enter {to_country}. "
            f"Visa type: {visa_type}. "
            f"Travel from {from_country} to {to_country} requires {visa_type} visa. "
            f"{from_country} citizens traveling to {to_country} must obtain visa."
        )
        
        return text
    
    def create_visa_id(self, from_country: str, to_country: str) -> str:
        """
        Create a unique ID for a visa requirement.
        
        Args:
            from_country: Source country
            to_country: Destination country
            
        Returns:
            Unique ID string
        """
        # Normalize country names and create ID
        from_normalized = from_country.lower().replace(' ', '_')
        to_normalized = to_country.lower().replace(' ', '_')
        return f"visa_{from_normalized}_to_{to_normalized}"
    
    # ========================================================================
    # EMBEDDING GENERATION
    # ========================================================================
    
    def embed_hotels(self, hotels: List[Dict[str, Any]], model_name: str = 'both') -> Dict[str, Any]:
        """
        Create embeddings for all hotels using specified model(s).
        
        Args:
            hotels: List of hotel dictionaries
            model_name: 'sbert', 'bge', or 'both'
            
        Returns:
            Dictionary mapping hotel_ids to their embeddings
        """
        embeddings_data = {
            'sbert': {},
            'bge': {}
        }
        
        print(f"\nCreating hotel embeddings for {len(hotels)} hotels...")
        
        for hotel in tqdm(hotels, desc="Embedding hotels"):
            # Create enhanced feature text (with review scores!)
            feature_text = self.create_hotel_feature_text(hotel)
            hotel_id = hotel['hotel_id']
            
            # Create embeddings with both models
            if model_name in ['sbert', 'both']:
                sbert_embedding = self.sbert_model.encode(feature_text)
                embeddings_data['sbert'][hotel_id] = sbert_embedding.tolist()
            
            if model_name in ['bge', 'both']:
                bge_embedding = self.bge_model.encode(feature_text)
                embeddings_data['bge'][hotel_id] = bge_embedding.tolist()
        
        print(f"✓ Created hotel embeddings for {len(hotels)} hotels")
        return embeddings_data
    
    def embed_visa_requirements(self, visa_reqs: List[Dict[str, Any]], model_name: str = 'both') -> Dict[str, Any]:
        """
        Create embeddings for all visa requirements using specified model(s).
        
        Args:
            visa_reqs: List of visa requirement dictionaries
            model_name: 'sbert', 'bge', or 'both'
            
        Returns:
            Dictionary mapping visa_ids to their embeddings
        """
        embeddings_data = {
            'sbert': {},
            'bge': {}
        }
        
        print(f"\nCreating visa embeddings for {len(visa_reqs)} visa requirements...")
        
        for visa_req in tqdm(visa_reqs, desc="Embedding visa requirements"):
            # Create feature text
            feature_text = self.create_visa_feature_text(visa_req)
            visa_id = self.create_visa_id(visa_req['from_country'], visa_req['to_country'])
            
            # Create embeddings with both models
            if model_name in ['sbert', 'both']:
                sbert_embedding = self.sbert_model.encode(feature_text)
                embeddings_data['sbert'][visa_id] = sbert_embedding.tolist()
            
            if model_name in ['bge', 'both']:
                bge_embedding = self.bge_model.encode(feature_text)
                embeddings_data['bge'][visa_id] = bge_embedding.tolist()
        
        print(f"✓ Created visa embeddings for {len(visa_reqs)} visa requirements")
        return embeddings_data
    
    # ========================================================================
    # STORAGE IN NEO4J
    # ========================================================================
    
    def store_hotel_embeddings_in_neo4j(self, embeddings_data: Dict[str, Any]):
        """
        Store hotel embeddings in Neo4j as properties on Hotel nodes.
        
        Args:
            embeddings_data: Dictionary with 'sbert' and 'bge' embeddings
        """
        print("\nStoring hotel embeddings in Neo4j...")
        
        # Store SBERT embeddings
        if embeddings_data['sbert']:
            print("Storing hotel SBERT embeddings...")
            with self.driver.session() as session:
                for hotel_id, embedding in tqdm(embeddings_data['sbert'].items(), 
                                                desc="Hotel SBERT"):
                    session.run("""
                        MATCH (h:Hotel {hotel_id: $hotel_id})
                        SET h.embedding_sbert = $embedding
                    """, hotel_id=hotel_id, embedding=embedding)
        
        # Store BGE embeddings
        if embeddings_data['bge']:
            print("Storing hotel BGE embeddings...")
            with self.driver.session() as session:
                for hotel_id, embedding in tqdm(embeddings_data['bge'].items(),
                                                desc="Hotel BGE"):
                    session.run("""
                        MATCH (h:Hotel {hotel_id: $hotel_id})
                        SET h.embedding_bge = $embedding
                    """, hotel_id=hotel_id, embedding=embedding)
        
        print("✓ All hotel embeddings stored in Neo4j")
    
    def store_visa_embeddings_in_neo4j(self, embeddings_data: Dict[str, Any], visa_reqs: List[Dict[str, Any]]):
        """
        Store visa embeddings in Neo4j as properties on NEEDS_VISA relationships.
        
        Args:
            embeddings_data: Dictionary with 'sbert' and 'bge' embeddings
            visa_reqs: Original visa requirement data (for from/to countries)
        """
        print("\nStoring visa embeddings in Neo4j...")
        
        # Create a mapping from visa_id to from/to countries
        visa_id_to_countries = {
            self.create_visa_id(v['from_country'], v['to_country']): v
            for v in visa_reqs
        }
        
        # Store SBERT embeddings
        if embeddings_data['sbert']:
            print("Storing visa SBERT embeddings...")
            with self.driver.session() as session:
                for visa_id, embedding in tqdm(embeddings_data['sbert'].items(),
                                               desc="Visa SBERT"):
                    countries = visa_id_to_countries[visa_id]
                    session.run("""
                        MATCH (from:Country {name: $from_country})-[v:NEEDS_VISA]->(to:Country {name: $to_country})
                        SET v.embedding_sbert = $embedding
                    """, from_country=countries['from_country'],
                        to_country=countries['to_country'],
                        embedding=embedding)
        
        # Store BGE embeddings
        if embeddings_data['bge']:
            print("Storing visa BGE embeddings...")
            with self.driver.session() as session:
                for visa_id, embedding in tqdm(embeddings_data['bge'].items(),
                                               desc="Visa BGE"):
                    countries = visa_id_to_countries[visa_id]
                    session.run("""
                        MATCH (from:Country {name: $from_country})-[v:NEEDS_VISA]->(to:Country {name: $to_country})
                        SET v.embedding_bge = $embedding
                    """, from_country=countries['from_country'],
                        to_country=countries['to_country'],
                        embedding=embedding)
        
        print("✓ All visa embeddings stored in Neo4j")
    
    # ========================================================================
    # VECTOR INDEXES
    # ========================================================================
    
    def create_vector_indexes(self):
        """
        Create vector indexes for both hotels and visa requirements.
        """
        print("\nCreating vector indexes...")
        
        with self.driver.session() as session:
            # Hotel SBERT index
            try:
                session.run("""
                    CREATE VECTOR INDEX hotel_sbert_index IF NOT EXISTS
                    FOR (h:Hotel)
                    ON h.embedding_sbert
                    OPTIONS {
                        indexConfig: {
                            `vector.dimensions`: 384,
                            `vector.similarity_function`: 'cosine'
                        }
                    }
                """)
                print("✓ Created hotel SBERT vector index (384 dimensions)")
            except Exception as e:
                print(f"Hotel SBERT index may already exist: {e}")
            
            # Hotel BGE index
            try:
                session.run("""
                    CREATE VECTOR INDEX hotel_bge_index IF NOT EXISTS
                    FOR (h:Hotel)
                    ON h.embedding_bge
                    OPTIONS {
                        indexConfig: {
                            `vector.dimensions`: 1024,
                            `vector.similarity_function`: 'cosine'
                        }
                    }
                """)
                print("✓ Created hotel BGE vector index (1024 dimensions)")
            except Exception as e:
                print(f"Hotel BGE index may already exist: {e}")
            
            # Note: Neo4j doesn't support vector indexes on relationships yet (as of Neo4j 5.x)
            # So visa embeddings will be stored but searched manually or via FAISS
            print("Note: Visa embeddings stored on relationships (no vector index on relationships yet)")
    
    # ========================================================================
    # VERIFICATION
    # ========================================================================
    
    def verify_embeddings(self, sample_size: int = 5):
        """Verify that embeddings were stored correctly."""
        print(f"\nVerifying embeddings...")
        
        # Check hotels
        print("\n--- Hotel Embeddings ---")
        hotel_query = """
        MATCH (h:Hotel)
        WHERE h.embedding_sbert IS NOT NULL AND h.embedding_bge IS NOT NULL
        RETURN h.hotel_id AS hotel_id,
               h.name AS hotel_name,
               size(h.embedding_sbert) AS sbert_dims,
               size(h.embedding_bge) AS bge_dims
        LIMIT $limit
        """
        
        with self.driver.session() as session:
            result = session.run(hotel_query, limit=sample_size)
            
            for record in result:
                print(f"✓ {record['hotel_name']}: "
                      f"SBERT={record['sbert_dims']}D, "
                      f"BGE={record['bge_dims']}D")
        
        # Count total hotels
        count_query = """
        MATCH (h:Hotel)
        WHERE h.embedding_sbert IS NOT NULL
        RETURN count(h) AS total
        """
        
        with self.driver.session() as session:
            result = session.run(count_query)
            total = result.single()['total']
            print(f"\n✓ Total hotels with embeddings: {total}")
        
        # Check visa requirements
        print("\n--- Visa Embeddings ---")
        visa_query = """
        MATCH (from:Country)-[v:NEEDS_VISA]->(to:Country)
        WHERE v.embedding_sbert IS NOT NULL AND v.embedding_bge IS NOT NULL
        RETURN from.name AS from_country,
               to.name AS to_country,
               size(v.embedding_sbert) AS sbert_dims,
               size(v.embedding_bge) AS bge_dims
        LIMIT $limit
        """
        
        with self.driver.session() as session:
            result = session.run(visa_query, limit=sample_size)
            
            for record in result:
                print(f"✓ {record['from_country']} → {record['to_country']}: "
                      f"SBERT={record['sbert_dims']}D, "
                      f"BGE={record['bge_dims']}D")
        
        # Count total visa requirements
        visa_count_query = """
        MATCH ()-[v:NEEDS_VISA]->()
        WHERE v.embedding_sbert IS NOT NULL
        RETURN count(v) AS total
        """
        
        with self.driver.session() as session:
            result = session.run(visa_count_query)
            total = result.single()['total']
            print(f"\n✓ Total visa requirements with embeddings: {total}")
    
    def close(self):
        """Close Neo4j connection."""
        self.driver.close()


# ============================================================================
# Main Execution
# ============================================================================

def main():
    """
    Main function to create and store ALL embeddings.
    """
    # Configuration
    NEO4J_URI = "neo4j://127.0.0.1:7687"
    NEO4J_USER = "neo4j"
    NEO4J_PASSWORD = "sSNrBXVgeDYG3y9Jv2pyLX_PgKolKhpv4ePEYChwHlw"
    
    # Initialize embedder
    embedder = EnhancedFeatureVectorEmbedder(
        neo4j_uri=NEO4J_URI,
        neo4j_user=NEO4J_USER,
        neo4j_password=NEO4J_PASSWORD
    )
    
    try:
        # ====================================================================
        # PART 1: HOTEL EMBEDDINGS (with review scores!)
        # ====================================================================
        print("\n" + "="*80)
        print("PART 1: CREATING HOTEL EMBEDDINGS")
        print("="*80)
        
        # Fetch hotels
        hotels = embedder.fetch_hotels_with_features()
        
        # Create embeddings
        hotel_embeddings = embedder.embed_hotels(hotels, model_name='both')
        
        # Store in Neo4j
        embedder.store_hotel_embeddings_in_neo4j(hotel_embeddings)
        
        # ====================================================================
        # PART 2: VISA EMBEDDINGS (new!)
        # ====================================================================
        print("\n" + "="*80)
        print("PART 2: CREATING VISA EMBEDDINGS")
        print("="*80)
        
        # Fetch visa requirements
        visa_reqs = embedder.fetch_visa_requirements()
        
        # Create embeddings
        visa_embeddings = embedder.embed_visa_requirements(visa_reqs, model_name='both')
        
        # Store in Neo4j
        embedder.store_visa_embeddings_in_neo4j(visa_embeddings, visa_reqs)
        
        # ====================================================================
        # PART 3: CREATE INDEXES & VERIFY
        # ====================================================================
        print("\n" + "="*80)
        print("PART 3: CREATING INDEXES & VERIFICATION")
        print("="*80)
        
        # Create vector indexes
        embedder.create_vector_indexes()
        
        # Verify all embeddings
        embedder.verify_embeddings(sample_size=5)
        
        print("\n" + "="*80)
        print("SUCCESS! All embeddings created and stored!")
        print("="*80)
        print("\nCreated embeddings for:")
        print("✓ Hotels (with review text AND scores)")
        print("✓ Visa requirements (country pairs)")
        print("\nYou can now use semantic similarity search for both!")
        
    finally:
        embedder.close()


if __name__ == "__main__":
    main()

Loading SBERT model...
Loading BGE model...
Models loaded successfully!

PART 1: CREATING HOTEL EMBEDDINGS
